# CIC-IDS-2017 data exploration

This notebook validates the merged CIC-IDS-2017 CSV before preprocessing or model training. The project-added `ClassLabel` column has been removed from the CSV; `Label` remains the source-of-truth target.

## Dataset feature dictionary

CICFlowMeter creates **bidirectional flows**. The first observed packet establishes the **forward** direction (`Src IP` → `Dst IP`); packets travelling in the reverse direction are **backward**. **Forward does not mean attacker → victim**, and backward does not mean victim → attacker. Those security roles must be inferred separately from the scenario, timestamp, IP addresses, and label. In this CIC-IDS-2017 export, time and inter-arrival-time fields are measured in microseconds, byte/length fields are measured in bytes, rate fields are per second, and count fields are dimensionless.

Definitions below follow the [official CICFlowMeter feature list](https://github.com/ahlashkari/CICFlowMeter/blob/master/ReadMe.txt), the [University of New Brunswick CICFlowMeter overview](https://www.unb.ca/cic/research/applications.html), and the published [`BasicFlow` implementation](https://github.com/ahlashkari/CICFlowMeter/blob/master/src/main/java/cic/cs/unb/ca/jnetpcap/BasicFlow.java). Dataset column spellings are preserved. Some behavior can vary between CICFlowMeter versions and is marked explicitly.

### What determines a flow?

Packets are grouped primarily by the network **5-tuple**:

1. source IP address;
2. destination IP address;
3. source port;
4. destination port;
5. IP protocol, such as TCP or UDP.

CICFlowMeter treats the reverse 5-tuple as the backward direction of the same bidirectional flow. For example, `192.168.10.5:51000 -> 104.16.10.20:443 (TCP)` and its reply `104.16.10.20:443 -> 192.168.10.5:51000 (TCP)` belong to the same flow. The first observed packet decides which direction is called forward.

Time also separates flows. TCP flows normally end on connection termination, such as FIN or RST, while UDP and inactive/long-running traffic are separated using configured timeouts. Reuse of the same 5-tuple after a flow ends can therefore produce a new flow. A more complete definition is:

> **Flow = bidirectional 5-tuple + its time/session boundary**

Attacker/victim roles and the benign/attack `Label` do not determine flow membership; they are interpretations assigned after packet grouping.

### Flow identity, endpoints, and target

| Column | Meaning |
|---|---|
| `Flow ID` | Identifier constructed from the flow endpoints, ports, and protocol. It identifies a flow inside the generated data but should not be assumed globally unique across captures. |
| `Src IP` | Source IP address of the first observed packet; establishes the forward-flow origin but is not necessarily the attacker. |
| `Src Port` | Transport-layer source port of the forward origin. A value of zero can occur when a port is not applicable. |
| `Dst IP` | Destination IP address of the first observed packet; establishes the forward-flow destination but is not necessarily the victim. |
| `Dst Port` | Transport-layer destination port. A value of zero can occur when a port is not applicable. |
| `Protocol` | Numeric IP protocol identifier, such as 6 for TCP and 17 for UDP. Treat it as categorical even though it is stored numerically. |
| `Timestamp` | Recorded start date and time of the flow. |
| `Label` | Ground-truth benign or attack class assigned to the flow. This is the supervised target and must never be included in model features. |

### Duration, traffic volume, and directional payload lengths

In ordinary language, **length** and **size** both mean a number of bytes. CICFlowMeter's names are legacy and not consistently distinct: the published source populates the packet `Length` and average `Size` statistics from each packet's **payload-byte count**, while `Header Length` separately measures header bytes. Therefore these fields should not be interpreted as complete on-the-wire frame sizes.

| Column | Meaning |
|---|---|
| `Flow Duration` | Elapsed time from the first to the last packet in the flow, in microseconds. |
| `Total Fwd Packet` | Number of packets travelling in the forward direction. |
| `Total Bwd packets` | Number of packets travelling in the backward direction. |
| `Total Length of Fwd Packet` | Sum of forward packet payload lengths, in bytes. It excludes the separately counted header bytes. |
| `Total Length of Bwd Packet` | Sum of backward packet payload lengths, in bytes. It excludes the separately counted header bytes. |
| `Fwd Packet Length Max` | Maximum forward packet payload length, in bytes. |
| `Fwd Packet Length Min` | Minimum forward packet payload length, in bytes. |
| `Fwd Packet Length Mean` | Mean forward packet payload length, in bytes. |
| `Fwd Packet Length Std` | Standard deviation of forward packet payload lengths, in bytes. |
| `Bwd Packet Length Max` | Maximum backward packet payload length, in bytes. |
| `Bwd Packet Length Min` | Minimum backward packet payload length, in bytes. |
| `Bwd Packet Length Mean` | Mean backward packet payload length, in bytes. |
| `Bwd Packet Length Std` | Standard deviation of backward packet payload lengths, in bytes. |

### Flow rates and inter-arrival times

`IAT` means **inter-arrival time**: the elapsed time between consecutive packets.

| Column | Meaning |
|---|---|
| `Flow Bytes/s` | Total forward and backward payload bytes divided by flow duration, expressed as bytes per second. |
| `Flow Packets/s` | Total flow packets divided by flow duration, expressed as packets per second. |
| `Flow IAT Mean` | Mean IAT across consecutive packets in the complete bidirectional flow, in microseconds. |
| `Flow IAT Std` | Standard deviation of complete-flow IATs, in microseconds. |
| `Flow IAT Max` | Maximum complete-flow IAT, in microseconds. |
| `Flow IAT Min` | Minimum complete-flow IAT, in microseconds. |
| `Fwd IAT Total` | Sum of IATs between consecutive forward packets, in microseconds. |
| `Fwd IAT Mean` | Mean forward-packet IAT, in microseconds. |
| `Fwd IAT Std` | Standard deviation of forward-packet IATs, in microseconds. |
| `Fwd IAT Max` | Maximum forward-packet IAT, in microseconds. |
| `Fwd IAT Min` | Minimum forward-packet IAT, in microseconds. |
| `Bwd IAT Total` | Sum of IATs between consecutive backward packets, in microseconds. |
| `Bwd IAT Mean` | Mean backward-packet IAT, in microseconds. |
| `Bwd IAT Std` | Standard deviation of backward-packet IATs, in microseconds. |
| `Bwd IAT Max` | Maximum backward-packet IAT, in microseconds. |
| `Bwd IAT Min` | Minimum backward-packet IAT, in microseconds. |

### Directional flags, headers, overall payload statistics, and TCP flags

`Fwd/Bwd PSH Flags` and `Fwd/Bwd URG Flags` are directional counters included by CICFlowMeter's legacy feature schema. There is no TCP rule requiring only PSH and URG to be directional; the tool simply does not export equivalent directional counters for every other flag. The later `... Flag Count` columns count flags across the complete bidirectional flow.

A directional PSH/URG value of **zero does not mean the flow is UDP**. It means CICFlowMeter counted no such flag in that direction. UDP flows also receive zero because TCP flags do not apply, but many TCP flows legitimately have zero PSH or URG flags.

| Column | Meaning |
|---|---|
| `Fwd PSH Flags` | Number of forward packets with the TCP PSH flag set. |
| `Bwd PSH Flags` | Number of backward packets with the TCP PSH flag set. |
| `Fwd URG Flags` | Number of forward packets with the TCP URG flag set. |
| `Bwd URG Flags` | Number of backward packets with the TCP URG flag set. |
| `Fwd Header Length` | Total bytes used by packet headers in the forward direction. |
| `Bwd Header Length` | Total bytes used by packet headers in the backward direction. |
| `Fwd Packets/s` | Forward packets per second. |
| `Bwd Packets/s` | Backward packets per second. |
| `Packet Length Min` | Minimum packet payload length across both directions, in bytes. |
| `Packet Length Max` | Maximum packet payload length across both directions, in bytes. |
| `Packet Length Mean` | Mean packet payload length across both directions, in bytes. |
| `Packet Length Std` | Standard deviation of packet payload lengths across both directions, in bytes. |
| `Packet Length Variance` | Variance of packet payload lengths across both directions, in squared bytes. |
| `FIN Flag Count` | Number of flow packets with the TCP FIN flag set. |
| `SYN Flag Count` | Number of flow packets with the TCP SYN flag set. |
| `RST Flag Count` | Number of flow packets with the TCP RST flag set. |
| `PSH Flag Count` | Number of flow packets with the TCP PSH flag set across both directions. |
| `ACK Flag Count` | Number of flow packets with the TCP ACK flag set. |
| `URG Flag Count` | Number of flow packets with the TCP URG flag set across both directions. |
| `CWR Flag Count` | Number of flow packets with the TCP CWR flag set. |
| `ECE Flag Count` | Number of flow packets with the TCP ECE flag set. |
| `Down/Up Ratio` | Backward packet count divided by forward packet count. In the published implementation the division is performed as integer division, so the fractional part is truncated: for example, 9 backward / 5 forward becomes 1 rather than 1.8. It is zero when no forward packet exists. |
| `Average Packet Size` | Average payload bytes per packet across the flow. Despite the word `Size`, it is derived from the same payload-length statistics. |
| `Fwd Segment Size Avg` | Average forward payload bytes per packet. In the published implementation this is mathematically the same quantity as `Fwd Packet Length Mean`. |
| `Bwd Segment Size Avg` | Average backward payload bytes per packet. In the published implementation this is mathematically the same quantity as `Bwd Packet Length Mean`. |

### Bulk transfers, subflows, and TCP window/segment features

These are CICFlowMeter-specific summaries rather than fields taken directly from a single network packet.

#### Bulk

A **bulk** is a sustained run of payload-carrying packets travelling in one direction. In the published CICFlowMeter implementation:

- only packets containing payload participate;
- packets must travel in the same direction;
- the candidate becomes a bulk when it reaches at least four packets;
- a gap longer than one second breaks the candidate bulk;
- traffic in the opposite direction can end or reset the candidate.

For example, four forward payload packets at `0.0 s`, `0.1 s`, `0.2 s`, and `0.3 s` form a forward bulk. Only three such packets do not form a bulk, and a gap greater than one second starts a new candidate. A zero bulk feature normally means no sequence satisfied these rules, not that the value is missing.

#### Subflow

A **subflow** is an activity chunk inside an existing flow. For example, a connection may send a burst of packets, pause for several seconds, and then send another burst while retaining the same 5-tuple. It remains one network flow, but CICFlowMeter can divide its activity into separate subflows. In the published implementation, an inactivity gap greater than one second creates a new subflow boundary.

```text
One bidirectional flow
|-- Subflow 1: first packet burst
`-- Subflow 2: second packet burst after an inactivity gap
```

The subflow features summarize average forward/backward packets and payload bytes per detected chunk. A subflow is not a new TCP connection. If a flow contains only one detected chunk, its subflow values can equal its whole-flow totals.

#### TCP receive window

The TCP **receive window** is a flow-control value advertised by an endpoint. It tells the peer approximately how much additional unacknowledged data the endpoint can currently receive. For example, an advertised window of `65,535` means the endpoint reports capacity for that amount of data; it does not mean that 65,535 bytes were actually transferred.

Each endpoint advertises its own receive window. When the first forward packet is a client SYN, the forward initial-window feature normally describes the client's receive capacity, while the backward feature normally describes the server's receive capacity. These values are not payload size, total transferred bytes, or the TCP congestion window. UDP has no TCP receive window, so an unavailable sentinel such as `-1` or `0` may appear depending on the extractor version.

#### Feature meanings

| Column | Meaning |
|---|---|
| `Fwd Bytes/Bulk Avg` | Average forward payload bytes per detected forward bulk. |
| `Fwd Packet/Bulk Avg` | Average packets per detected forward bulk transfer. |
| `Fwd Bulk Rate Avg` | Total forward bulk payload bytes divided by total forward bulk duration, in bytes per second. |
| `Bwd Bytes/Bulk Avg` | Average backward payload bytes per detected backward bulk. |
| `Bwd Packet/Bulk Avg` | Average packets per detected backward bulk transfer. |
| `Bwd Bulk Rate Avg` | Total backward bulk payload bytes divided by total backward bulk duration, in bytes per second. |
| `Subflow Fwd Packets` | Average number of forward packets per detected subflow. |
| `Subflow Fwd Bytes` | Average number of forward bytes per detected subflow. |
| `Subflow Bwd Packets` | Average number of backward packets per detected subflow. |
| `Subflow Bwd Bytes` | Average number of backward bytes per detected subflow. |
| `FWD Init Win Bytes` | TCP window value reported for the initial forward packet. It is an advertised receive-window field, not transferred bytes. In this dataset, `-1` is an unavailable/not-applicable sentinel. |
| `Bwd Init Win Bytes` | TCP window value reported for the backward direction. It is an advertised receive-window field, not transferred bytes. In this dataset, `-1` is an unavailable/not-applicable sentinel. |
| `Fwd Act Data Pkts` | Number of forward packets carrying at least one byte of TCP payload. |
| `Fwd Seg Size Min` | Minimum forward packet header-byte count in the published implementation. Despite the name, it is not the minimum forward payload size. |

### Active and idle periods

CICFlowMeter divides sufficiently long flows into alternating **active** and **idle** periods according to its activity timeout. These statistics use microseconds in this export.

| Column | Meaning |
|---|---|
| `Active Mean` | Mean duration of active periods. |
| `Active Std` | Standard deviation of active-period durations. |
| `Active Max` | Maximum active-period duration. |
| `Active Min` | Minimum active-period duration. |
| `Idle Mean` | Mean duration of idle periods. |
| `Idle Std` | Standard deviation of idle-period durations. |
| `Idle Max` | Maximum idle-period duration. |
| `Idle Min` | Minimum idle-period duration. |

## 1. Imports and dataset path

In [25]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

dataset_candidates = [
    Path("../data/raw/cicids2017_merged.csv"),
    Path("ml/data/raw/cicids2017_merged.csv"),
]

DATASET_PATH = next(
    (path.resolve() for path in dataset_candidates if path.exists()),
    None,
)

if DATASET_PATH is None:
    raise FileNotFoundError(
        "Could not find ml/data/raw/cicids2017_merged.csv. "
        "Run the notebook from the repository root or ml/notebooks."
    )

print(f"Dataset: {DATASET_PATH}")
print(f"File size: {DATASET_PATH.stat().st_size / (1024 ** 3):.2f} GiB")

Dataset: C:\Users\ademz\Desktop\9raya\DoS-Intrusion-Detection-System\ml\data\raw\cicids2017_merged.csv
File size: 1.45 GiB


## 2. Load the dataset

The merged CSV is large, so this cell may take some time and require several gigabytes of memory.

In [26]:
df = pd.read_csv(DATASET_PATH, low_memory=False)

if "ClassLabel" in df.columns:
    raise ValueError("ClassLabel should not be present in the raw dataset.")

print("Dataset loaded successfully.")

Dataset loaded successfully.


## 3. Dataset dimensions and columns

In [27]:
row_count, column_count = df.shape

print(f"Rows: {row_count:,}")
print(f"Columns: {column_count}")
print("\nColumn names:")

for index, column in enumerate(df.columns, start=1):
    print(f"{index}. {column}")

Rows: 3,119,345
Columns: 84

Column names:
1. Flow ID
2. Src IP
3. Src Port
4. Dst IP
5. Dst Port
6. Protocol
7. Timestamp
8. Flow Duration
9. Total Fwd Packet
10. Total Bwd packets
11. Total Length of Fwd Packet
12. Total Length of Bwd Packet
13. Fwd Packet Length Max
14. Fwd Packet Length Min
15. Fwd Packet Length Mean
16. Fwd Packet Length Std
17. Bwd Packet Length Max
18. Bwd Packet Length Min
19. Bwd Packet Length Mean
20. Bwd Packet Length Std
21. Flow Bytes/s
22. Flow Packets/s
23. Flow IAT Mean
24. Flow IAT Std
25. Flow IAT Max
26. Flow IAT Min
27. Fwd IAT Total
28. Fwd IAT Mean
29. Fwd IAT Std
30. Fwd IAT Max
31. Fwd IAT Min
32. Bwd IAT Total
33. Bwd IAT Mean
34. Bwd IAT Std
35. Bwd IAT Max
36. Bwd IAT Min
37. Fwd PSH Flags
38. Bwd PSH Flags
39. Fwd URG Flags
40. Bwd URG Flags
41. Fwd Header Length
42. Bwd Header Length
43. Fwd Packets/s
44. Bwd Packets/s
45. Packet Length Min
46. Packet Length Max
47. Packet Length Mean
48. Packet Length Std
49. Packet Length Variance
50. FIN

## 4. Duplicate column names

Check this before selecting or transforming columns by name.

In [28]:
duplicate_columns = df.columns[df.columns.duplicated()].tolist()

print(f"Duplicate column names: {len(duplicate_columns)}")
print(duplicate_columns if duplicate_columns else "No duplicate column names found.")

Duplicate column names: 0
No duplicate column names found.


## 5. First rows

In [29]:
display(df.head())

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.10.5-104.16.207.165-54865-443-6,104.16.207.165,443.0,192.168.10.5,54865.0,6.0,7/7/2017 3:30,3.0,2.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,192.168.10.5-104.16.28.216-55054-80-6,104.16.28.216,80.0,192.168.10.5,55054.0,6.0,7/7/2017 3:30,109.0,1.0,1.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,192.168.10.5-104.16.28.216-55055-80-6,104.16.28.216,80.0,192.168.10.5,55055.0,6.0,7/7/2017 3:30,52.0,1.0,1.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,192.168.10.16-104.17.241.25-46236-443-6,104.17.241.25,443.0,192.168.10.16,46236.0,6.0,7/7/2017 3:30,34.0,1.0,1.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,192.168.10.5-104.19.196.102-54863-443-6,104.19.196.102,443.0,192.168.10.5,54863.0,6.0,7/7/2017 3:30,3.0,2.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


## 6. Data types and memory usage

In [30]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3119345 entries, 0 to 3119344
Data columns (total 84 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Flow ID                     object 
 1   Src IP                      object 
 2   Src Port                    float64
 3   Dst IP                      object 
 4   Dst Port                    float64
 5   Protocol                    float64
 6   Timestamp                   object 
 7   Flow Duration               float64
 8   Total Fwd Packet            float64
 9   Total Bwd packets           float64
 10  Total Length of Fwd Packet  float64
 11  Total Length of Bwd Packet  float64
 12  Fwd Packet Length Max       float64
 13  Fwd Packet Length Min       float64
 14  Fwd Packet Length Mean      float64
 15  Fwd Packet Length Std       float64
 16  Bwd Packet Length Max       float64
 17  Bwd Packet Length Min       float64
 18  Bwd Packet Length Mean      float64
 19  Bwd Packet Length Std

## 7. Validate the target label

A flow without `Label` cannot be used for supervised learning. Check missing labels first and confirm whether those rows contain any useful feature values.

In [31]:
invalid_label_mask = df["Label"].isna()
invalid_row_count = int(invalid_label_mask.sum())
invalid_row_percentage = invalid_row_count / len(df) * 100

print(f"Rows without Label: {invalid_row_count:,} ({invalid_row_percentage:.2f}%)")
display(df.loc[invalid_label_mask].head())

invalid_non_null_counts = df.loc[invalid_label_mask].notna().sum()
invalid_non_null_counts = invalid_non_null_counts[
    invalid_non_null_counts > 0
].sort_values(ascending=False)

if invalid_non_null_counts.empty:
    print("All rows without Label are completely empty across the 84 columns.")
else:
    print("Non-null values found in rows without Label:")
    display(invalid_non_null_counts.to_frame("non_null_count"))

Rows without Label: 288,602 (9.25%)


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
1692131,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1692132,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1692133,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1692134,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1692135,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


All rows without Label are completely empty across the 84 columns.


## 8. Create the valid working dataset

This changes only the in-memory DataFrame. It does not modify the CSV.

In [32]:
rows_before = len(df)
df.dropna(subset=["Label"], inplace=True)
df.reset_index(drop=True, inplace=True)
df_valid = df
del df
del invalid_label_mask

print(f"Rows before: {rows_before:,}")
print(f"Rows removed: {rows_before - len(df_valid):,}")
print(f"Valid rows remaining: {len(df_valid):,}")

Rows before: 3,119,345
Rows removed: 288,602
Valid rows remaining: 2,830,743


## 9. Detailed label distribution

Inspect the target classes only after removing rows that have no usable target.

In [33]:
label_counts = df_valid["Label"].value_counts(dropna=False)
label_distribution = pd.DataFrame(
    {
        "count": label_counts,
        "percentage": (label_counts / len(df_valid) * 100).round(6),
    }
)

print(f"Unique detailed labels: {df_valid['Label'].nunique(dropna=False)}")
display(label_distribution)

Unique detailed labels: 15


,count,percentage
Label,,
BENIGN,2273097,80.300366
DoS Hulk,231073,8.162981
PortScan,158930,5.614427
DDoS,128027,4.522735
DoS GoldenEye,10293,0.363615
FTP-Patator,7938,0.280421
SSH-Patator,5897,0.208320
DoS slowloris,5796,0.204752
DoS Slowhttptest,5499,0.194260


## 10. Missing feature values before infinity normalization

This records the values that were originally missing, before infinity is converted to `NaN`.

In [34]:
feature_columns = df_valid.columns.drop("Label")
missing_before = df_valid.loc[:, feature_columns].isna().sum()
missing_before_summary = pd.DataFrame(
    {
        "missing_count": missing_before,
        "missing_percentage": (missing_before / len(df_valid) * 100).round(6),
    }
).sort_values("missing_count", ascending=False)

print(f"Originally missing feature values: {int(missing_before.sum()):,}")
display(missing_before_summary[missing_before_summary["missing_count"] > 0])

Originally missing feature values: 1,358


,missing_count,missing_percentage
Flow Bytes/s,1358,0.047973


## 11. Inspect infinite feature values

Count infinite values without changing the dataset.

In [35]:
numeric_columns = df_valid.select_dtypes(include="number").columns
infinite_counts = pd.Series(
    {
        column: int(np.isinf(df_valid[column].to_numpy()).sum())
        for column in numeric_columns
    },
    name="infinite_count",
).sort_values(ascending=False)

infinite_counts = infinite_counts[infinite_counts > 0]
print(f"Columns containing infinity: {len(infinite_counts)}")
print(f"Total infinite values: {int(infinite_counts.sum()):,}")
display(infinite_counts.to_frame())

Columns containing infinity: 2
Total infinite values: 4,376


,infinite_count
Flow Packets/s,2867
Flow Bytes/s,1509


## 12. Normalize infinite values

Convert positive and negative infinity to `NaN` so every non-finite feature value has one consistent representation.

In [36]:
affected_columns = infinite_counts.index.tolist()

if affected_columns:
    df_valid.loc[:, affected_columns] = df_valid[affected_columns].replace(
        [np.inf, -np.inf],
        np.nan,
    )

print(f"Infinite values converted to NaN: {int(infinite_counts.sum()):,}")

Infinite values converted to NaN: 4,376


## 13. Missing feature values after infinity normalization

This is the final missing-value pool that preprocessing must handle.

In [37]:
missing_after = df_valid.loc[:, feature_columns].isna().sum()
missing_after_summary = pd.DataFrame(
    {
        "missing_count": missing_after,
        "missing_percentage": (missing_after / len(df_valid) * 100).round(6),
    }
).sort_values("missing_count", ascending=False)

print(f"Missing feature values after normalization: {int(missing_after.sum()):,}")
display(missing_after_summary[missing_after_summary["missing_count"] > 0])

Missing feature values after normalization: 5,734


,missing_count,missing_percentage
Flow Bytes/s,2867,0.101281
Flow Packets/s,2867,0.101281


## 14. Numeric feature descriptive statistics

Inspect distributions after infinity normalization. Transposing the output makes the feature-level summary easier to read.

In [40]:
numeric_summary = df_valid.loc[:, numeric_columns].describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
).T

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(numeric_summary)

,count,mean,std,min,1%,25%,50%,75%,99%,max
Src Port,2830743.0,4.112886e+04,2.229494e+04,0.000000e+00,80.000000,32774.000000,50944.000000,5.841300e+04,6.490800e+04,6.553500e+04
Dst Port,2830743.0,8.071483e+03,1.828363e+04,0.000000e+00,22.000000,53.000000,80.000000,4.430000e+02,6.165200e+04,6.553500e+04
Protocol,2830743.0,9.880341e+00,5.261922e+00,0.000000e+00,6.000000,6.000000,6.000000,1.700000e+01,1.700000e+01,1.700000e+01
Flow Duration,2830743.0,1.478566e+07,3.365374e+07,-1.300000e+01,1.000000,155.000000,31316.000000,3.204828e+06,1.177776e+08,1.200000e+08
Total Fwd Packet,2830743.0,9.361160e+00,7.496728e+02,1.000000e+00,1.000000,2.000000,2.000000,5.000000e+00,4.800000e+01,2.197590e+05
Total Bwd packets,2830743.0,1.039377e+01,9.973883e+02,0.000000e+00,0.000000,1.000000,2.000000,4.000000e+00,5.700000e+01,2.919220e+05
Total Length of Fwd Packet,2830743.0,5.493024e+02,9.993589e+03,0.000000e+00,0.000000,12.000000,62.000000,1.870000e+02,1.159500e+04,1.290000e+07
Total Length of Bwd Packet,2830743.0,1.616264e+04,2.263088e+06,0.000000e+00,0.000000,0.000000,123.000000,4.820000e+02,7.183100e+04,6.554530e+08
Fwd Packet Length Max,2830743.0,2.075999e+02,7.171848e+02,0.000000e+00,0.000000,6.000000,37.000000,8.100000e+01,2.936000e+03,2.482000e+04
Fwd Packet Length Min,2830743.0,1.871366e+01,6.033935e+01,0.000000e+00,0.000000,0.000000,2.000000,3.600000e+01,7.700000e+01,2.325000e+03


## 15. Zero and negative flow durations

Zero duration can represent a single-packet or sub-resolution flow. Negative duration is invalid. Measure them separately and inspect their target classes.

In [46]:
zero_duration_mask = df_valid["Flow Duration"] == 0
negative_duration_mask = df_valid["Flow Duration"] < 0
zero_duration_count = int(zero_duration_mask.sum())
negative_duration_count = int(negative_duration_mask.sum())

duration_summary = pd.DataFrame(
    {
        "count": [zero_duration_count, negative_duration_count],
        "percentage": [
            zero_duration_mask.mean() * 100,
            negative_duration_mask.mean() * 100,
        ],
    },
    index=["zero_duration", "negative_duration"],
)
display(duration_summary)

for duration_type, mask in {
    "zero_duration": zero_duration_mask,
    "negative_duration": negative_duration_mask,
}.items():
    print(f"\nLabel distribution for {duration_type}:")
    display(
        df_valid.loc[mask, "Label"]
        .value_counts()
        .to_frame("flow_count")
    )

,count,percentage
zero_duration,2867,0.101281
negative_duration,115,0.004063



Label distribution for zero_duration:


,flow_count
Label,
BENIGN,1777
DoS Hulk,949
PortScan,126
Bot,10
FTP-Patator,3
DDoS,2



Label distribution for negative_duration:


,flow_count
Label,
BENIGN,115


## 16. Systematic negative-value and TCP-window validation

Every numeric feature should be non-negative except the two TCP-window fields, where `-1` is an unavailable/not-applicable sentinel. Check all other numeric features systematically, then validate the window fields separately: accepted values are `-1` or the unsigned 16-bit range `0..65535`.

In [47]:
window_sentinel_features = ["FWD Init Win Bytes", "Bwd Init Win Bytes"]
non_negative_features = [
    feature
    for feature in numeric_columns
    if feature not in window_sentinel_features
]

negative_counts = pd.Series(
    {
        feature: int((df_valid[feature] < 0).sum())
        for feature in non_negative_features
    },
    name="negative_count",
)
negative_counts = negative_counts[negative_counts > 0].sort_values(ascending=False)

if negative_counts.empty:
    print("No unexpected negative values found.")
else:
    negative_summary = pd.DataFrame(
        {
            "negative_count": negative_counts,
            "percentage": (negative_counts / len(df_valid) * 100).round(6),
            "minimum": [
                df_valid[feature].min()
                for feature in negative_counts.index
            ],
        }
    )
    display(negative_summary)

    negative_label_distribution = pd.concat(
        {
            feature: df_valid.loc[
                df_valid[feature] < 0,
                "Label",
            ].value_counts()
            for feature in negative_counts.index
        },
        names=["feature", "Label"],
    ).rename("count").to_frame()

    negative_label_distribution["percentage_within_feature"] = (
        negative_label_distribution["count"]
        / negative_label_distribution.groupby(level="feature")["count"].transform("sum")
        * 100
    ).round(4)

    with pd.option_context("display.max_rows", None):
        display(negative_label_distribution)

window_validation_records = []
invalid_window_label_records = []

for feature in window_sentinel_features:
    values = df_valid[feature]
    invalid_mask = (values < -1) | (values > 65535)
    invalid_count = int(invalid_mask.sum())

    window_validation_records.append(
        {
            "feature": feature,
            "minimum": values.min(),
            "maximum": values.max(),
            "sentinel_minus_one_count": int((values == -1).sum()),
            "invalid_count": invalid_count,
            "invalid_percentage": invalid_count / len(df_valid) * 100,
        }
    )

    if invalid_count:
        label_counts_for_invalid = df_valid.loc[
            invalid_mask,
            "Label",
        ].value_counts()
        for label, count in label_counts_for_invalid.items():
            invalid_window_label_records.append(
                {"feature": feature, "Label": label, "count": int(count)}
            )

window_validation_summary = pd.DataFrame(window_validation_records).set_index("feature")
display(window_validation_summary)

if invalid_window_label_records:
    display(pd.DataFrame(invalid_window_label_records))
else:
    print("No TCP-window values below -1 or above 65535.")

,negative_count,percentage,minimum
Flow IAT Min,2891,0.102129,-1.400000e+01
Flow Duration,115,0.004063,-1.300000e+01
Flow IAT Mean,115,0.004063,-1.300000e+01
Flow Packets/s,115,0.004063,-2.000000e+06
Flow IAT Max,115,0.004063,-1.300000e+01
Flow Bytes/s,85,0.003003,-2.610000e+08
Fwd Header Length,35,0.001236,-3.221223e+10
Fwd Seg Size Min,35,0.001236,-5.368707e+08
Bwd Header Length,22,0.000777,-1.073741e+09
Fwd IAT Min,17,0.000601,-1.200000e+01


count  percentage_within_feature
feature           Label                                          
Flow IAT Min      BENIGN          2697                    93.2895
                  DoS Hulk         159                     5.4998
                  DDoS              19                     0.6572
                  DoS GoldenEye      5                     0.1730
                  Heartbleed         4                     0.1384
                  FTP-Patator        4                     0.1384
                  SSH-Patator        2                     0.0692
                  Infiltration       1                     0.0346
Flow Duration     BENIGN           115                   100.0000
Flow IAT Mean     BENIGN           115                   100.0000
Flow Packets/s    BENIGN           115                   100.0000
Flow IAT Max      BENIGN           115                   100.0000
Flow Bytes/s      BENIGN            85                   100.0000
Fwd Header Length BENIGN            35                   100.0000
Fwd Seg Size Min  BENIGN            35                   100.0000
Bwd Header Length BENIGN            22                   100.0000
Fwd IAT Min       DoS Hulk           8                    47.0588
                  DDoS               6                    35.2941
                  DoS GoldenEye      3                    17.6471

,minimum,maximum,sentinel_minus_one_count,invalid_count,invalid_percentage
feature,,,,,
FWD Init Win Bytes,-1.0,65535.0,1001189,0,0.0
Bwd Init Win Bytes,-1.0,65535.0,1441552,0,0.0


No TCP-window values below -1 or above 65535.


## 17. Exact constant columns

Constant features contain no information for classification. Detect them exactly without deleting them.

In [44]:
constant_records = []

for feature in feature_columns:
    values = df_valid[feature]
    first_value = values.iloc[0]
    is_constant = (
        values.isna().all()
        if pd.isna(first_value)
        else values.eq(first_value).all()
    )

    if is_constant:
        constant_records.append(
            {"feature": feature, "constant_value": first_value}
        )

constant_columns = pd.DataFrame(constant_records)
print(f"Exact constant columns: {len(constant_columns)}")
display(constant_columns)

Exact constant columns: 8


,feature,constant_value
0,Bwd PSH Flags,0.0
1,Bwd URG Flags,0.0
2,Fwd Bytes/Bulk Avg,0.0
3,Fwd Packet/Bulk Avg,0.0
4,Fwd Bulk Rate Avg,0.0
5,Bwd Bytes/Bulk Avg,0.0
6,Bwd Packet/Bulk Avg,0.0
7,Bwd Bulk Rate Avg,0.0


## 18. Suspected redundant feature pairs

Compare only conceptually related pairs row-by-row. Matching summary statistics alone do not prove redundancy, and semantically unrelated TCP flags are intentionally excluded.

In [48]:
semantically_related_pairs = [
    ("Fwd Packet Length Mean", "Fwd Segment Size Avg"),
    ("Bwd Packet Length Mean", "Bwd Segment Size Avg"),
    ("Total Fwd Packet", "Subflow Fwd Packets"),
    ("Total Bwd packets", "Subflow Bwd Packets"),
    ("Total Length of Fwd Packet", "Subflow Fwd Bytes"),
    ("Total Length of Bwd Packet", "Subflow Bwd Bytes"),
]

redundancy_records = []

for left_feature, right_feature in semantically_related_pairs:
    left_values = df_valid[left_feature]
    right_values = df_valid[right_feature]
    equal_mask = left_values.eq(right_values) | (
        left_values.isna() & right_values.isna()
    )
    different_rows = int((~equal_mask).sum())
    max_absolute_difference = (left_values - right_values).abs().max()

    redundancy_records.append(
        {
            "left_feature": left_feature,
            "right_feature": right_feature,
            "exactly_identical": different_rows == 0,
            "different_rows": different_rows,
            "different_percentage": different_rows / len(df_valid) * 100,
            "max_absolute_difference": max_absolute_difference,
        }
    )

redundancy_summary = pd.DataFrame(redundancy_records)
display(redundancy_summary)

,left_feature,right_feature,exactly_identical,different_rows,different_percentage,max_absolute_difference
0,Fwd Packet Length Mean,Fwd Segment Size Avg,False,53484,1.889398,1.818989e-12
1,Bwd Packet Length Mean,Bwd Segment Size Avg,False,36936,1.304816,1.000000e-06
2,Total Fwd Packet,Subflow Fwd Packets,True,0,0.000000,0.000000e+00
3,Total Bwd packets,Subflow Bwd Packets,True,0,0.000000,0.000000e+00
4,Total Length of Fwd Packet,Subflow Fwd Bytes,False,1,0.000035,2.966200e+04
5,Total Length of Bwd Packet,Subflow Bwd Bytes,False,146,0.005158,4.904610e+05


## Findings to record before preprocessing

After running all cells, record:

- whether every row missing `Label` is completely empty;
- the valid target classes and their imbalance;
- which feature values were originally missing;
- which features contained infinity and how many values were normalized;
- the final missing-value pool after normalization;
- which numeric features have extreme ranges or highly skewed distributions;
- how many zero and negative-duration flows exist and which labels they belong to;
- which numeric features contain unexpected negative values and whether TCP-window values violate their allowed range;
- which feature columns are exactly constant;
- which suspected redundant pairs are genuinely identical;
- the exact spelling and encoding of every detailed label;
- which rare attack classes need special handling.